### model.py


This script defines a continuous generative prior over approximately phylogenetic distance matrices by modeling latent taxa positions within a smooth geometric space, bypassing the need to explicitly sample discrete tree topologies. The pipeline starts in Euclidean space by sampling unconstrained coordinate vectors for each leaf taxon from a standard Gaussian distribution. These flat vectors are then radially projected into the open unit disk of the Poincaré ball model of hyperbolic space using a stable tangent hyperbolic mapping, a geometry uniquely optimized for capturing hierarchical branching patterns due to its negative curvature. Once positioned, the pairwise hyperbolic distances between all taxa are computed to construct a raw distance metric. To ensure that global scale artifacts do not interfere with structural training, the upper triangle of this matrix is isolated to compute a stable mean, and the entire matrix is normalized to produce D_tilde, fixing the average pairwise distance strictly at $1.0$.  



The relative tree-like structure of this normalized space is then regularized using two simultaneous, differentiable geometric forces. The first force, controlled by lmbda_4, applies a smooth, temperature-scaled softplus relaxation of the classical four-point condition across a fixed subset of training quartets; this penalizes instances where one cross-distance sum uniquely dominates the others, effectively bending the latent coordinate cloud toward an additive tree-metric structure. The second force, controlled by lmbda_g, calculates the quartet gap—the difference between the middle and smallest sorted quartet sums—and penalizes its squared deviation from a target minimum branch length g0. This acts as an essential safeguard that pushes internal nodes apart, preventing the four-point penalty from lazily collapsing the geometry into an uninformative, star-like tree failure mode. Finally, the framework offers an optional scale reconstruction step: if absolute distances are required downstream, a positive scalar multiplier s is sampled from a log-normal distribution to globally stretch or shrink the tree, mapping the relative patristic geometry back onto a biologically or representationally meaningful data scale.  

$$\mathcal{U}(U) = -\log p_0(U) + \lambda_4 \hat{P}_{4pt}(\tilde{D}) + \lambda_g \hat{P}_{gap}(\tilde{D})$$

### penalty.py

This script acts as the core mathematical optimization engine for the framework, managing both the uniform Monte Carlo sampling of taxonomic quartets and the fully vectorized calculation of smooth tree-metric penalties. The script begins by using a fixed random seed to draw a localized subset of training quartets from the total pool of $N$ leaves without ever having to materialize the memory-restrictive combinatorial space of all $\binom{N}{4}$ configurations. To allow for an honest, unbiased post-hoc evaluation of the model's structural progress, a separate function builds a pristine validation set of test quartets by checking newly sampled indices against a high-speed lookup hash set containing the training duplicates, effectively eliminating any risk of data leakage. Once these index pathways are established, the soft_four_point_penalty function processes all training quartets simultaneously by extracting their index coordinates to compute the three unique cross-distance combinations ($s_1, s_2, s_3$) that define a quartet's split geometry. 

The final step uses an elegant broadcasting matrix subtraction array to calculate every cross-difference between these three distance sums in a highly efficient $3 \times 3$ grid per quartet, bypassing slow native Python loops. These raw differences are then passed through a temperature-scaled softplus function ($\zeta_\tau$), a continuous relaxation that acts as a smooth alternative to a hard maximum operator to guarantee stable gradient paths for the downstream NUTS sampler. Because a sum subtracted from itself yields zero, and $\zeta_\tau(0)$ evaluates to a non-zero residue that would corrupt an active product string, the code applies a specialized identity matrix mask. This mask completely clears the diagonal differences to zero and inserts a neutral value of $1.0$ precisely along the diagonal of the grid, ensuring that when PyTorch multiplies elements across rows and sums across columns, the identity entries are mathematically skipped. The resulting 1D output vector contains a differentiable penalty value for each quartet that scales sharply only if a single cross-sum uniquely dominates the other two, providing the explicit structural pressure required to bend unconstrained latent representations toward an additive, phylogenetic tree-like structure. 

$$p(q;D) = \sum_{k=1}^{3} \prod_{j \neq k} \zeta_{\tau}(s_k - s_j)$$

### geometry.py

This script functions as the spatial framework of the repository, translating unconstrained latent configurations into a complete pairwise distance matrix within the curved geometry of the Poincaré ball model of hyperbolic space. To maintain high performance across both batched MCMC samples and unbatched steps, the script avoids slow python loops by using matrix expansions to broadcast coordinate pairs against each other simultaneously. It computes the squared Euclidean distance matrix between all points alongside a boundary-distance denominator field derived from each node's radial distance from the origin. By applying strict clipping boundaries (torch.clamp) to neutralize division-by-zero anomalies and out-of-domain inputs, the script safely processes the ratio of these values through an inverse hyperbolic cosine function (torch.acosh). The resulting output matrix satisfies all necessary metric parameters while naturally scaling distances exponentially near the boundary disk, providing the foundation required to model hierarchical branching trees without requiring explicit topology states.  

$$d_{\mathbb{H}}(x_i, x_j) = \text{arcosh}\left(1 + 2\frac{\|x_i - x_j\|_2^2}{(1 - \|x_i\|_2^2)(1 - \|x_j\|_2^2)}\right)$$